### Exatract Phase

In [2]:
import pandas as pd
customers = pd.read_csv("D:/Projects/customers.csv") 
customers

,customer_id,first_name,last_name,city,age,email
0,1,Ahmed,Hassan,Cairo,25,ahmed@email.com
1,2,Sara,Ali,Giza,31,sara@email.com
2,3,Omar,Mohamed,cairo,17,omar@email.com
3,4,Mona,Ibrahim,Alexandria,28,NaN
4,5,Ahmed,Hassan,Cairo,25,ahmed@email.com


In [3]:
orders = pd.read_parquet("D:/Projects/orders.parquet")
orders

,order_id,customer_id,product_id,quantity,order_date
0,1001,1,101,2,2026-09-01
1,1002,2,102,1,2026-09-01
2,1003,3,103,3,2026-09-02
3,1004,4,101,1,2026-09-02
4,1005,1,104,2,2026-09-03
5,1006,5,102,1,2026-09-03
6,1007,2,105,4,2026-09-04
7,1008,3,101,1,2026-09-04
8,1009,4,103,2,2026-09-05
9,1010,1,104,1,2026-09-05


In [4]:
import requests
import pandas as pd 

url = "https://raw.githubusercontent.com/MohammedHameds/test1455/refs/heads/main/products.csv"

products = pd.read_csv(url)

products.to_csv("products.csv", index=False)

### Transformation Phase

In [5]:
customers

,customer_id,first_name,last_name,city,age,email
0,1,Ahmed,Hassan,Cairo,25,ahmed@email.com
1,2,Sara,Ali,Giza,31,sara@email.com
2,3,Omar,Mohamed,cairo,17,omar@email.com
3,4,Mona,Ibrahim,Alexandria,28,NaN
4,5,Ahmed,Hassan,Cairo,25,ahmed@email.com


In [6]:
customers = customers.drop_duplicates()
customers['city'] = customers['city'].replace({'cairo':'Cairo'}) 
customers['age'] = pd.to_numeric(customers["age"])
customers['age'] > 0
customers['email'] = customers['email'].fillna("Email Not Found")
customers["full_name"] = customers["first_name"] + " " + customers["last_name"]

In [7]:
orders = orders.drop_duplicates()
orders['quantity'] = pd.to_numeric(orders['quantity'])
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')

In [8]:
orders['quantity'] > 0

0    True
1    True
2    True
3    True
4    True
5    True
6    True
7    True
8    True
9    True
Name: quantity, dtype: bool

In [9]:
products

,product_id,product_name,category,unit_price
0,101,Laptop,Electronics,25000
1,102,Mouse,Accessories,500
2,103,Keyboard,Accessories,1200
3,104,Monitor,Electronics,7000
4,105,Headphones,Accessories,2500
5,106,Webcam,Accessories,1800


In [10]:
products.drop_duplicates() 
products['unit_price'] = products['unit_price'].astype('float64')

In [11]:
products['unit_price'] > 0

0    True
1    True
2    True
3    True
4    True
5    True
Name: unit_price, dtype: bool

In [12]:
Sales = pd.merge(pd.merge(customers, orders, on='customer_id'), products, on='product_id')
print(Sales)

   customer_id first_name last_name        city  age            email  \
0            1      Ahmed    Hassan       Cairo   25  ahmed@email.com   
1            1      Ahmed    Hassan       Cairo   25  ahmed@email.com   
2            1      Ahmed    Hassan       Cairo   25  ahmed@email.com   
3            2       Sara       Ali        Giza   31   sara@email.com   
4            2       Sara       Ali        Giza   31   sara@email.com   
5            3       Omar   Mohamed       Cairo   17   omar@email.com   
6            3       Omar   Mohamed       Cairo   17   omar@email.com   
7            4       Mona   Ibrahim  Alexandria   28  Email Not Found   
8            4       Mona   Ibrahim  Alexandria   28  Email Not Found   
9            5      Ahmed    Hassan       Cairo   25  ahmed@email.com   

      full_name  order_id  product_id  quantity order_date product_name  \
0  Ahmed Hassan      1001         101         2 2026-09-01       Laptop   
1  Ahmed Hassan      1005         104         

In [13]:
Sales["total_amount"] = Sales["quantity"] * Sales["unit_price"]

In [14]:
Sales = Sales[['order_id', 'customer_id', 'product_id', 'quantity', 'unit_price', 'total_amount']] 

In [15]:
df = pd.DataFrame(Sales)
df.to_csv('Sales.csv', index=False)

### Load (SQL Server) Phase

In [16]:
import pyodbc

In [17]:
data = list(Sales.itertuples(index = False, name = None))
data

[(1001, 1, 101, 2, 25000.0, 50000.0),
 (1005, 1, 104, 2, 7000.0, 14000.0),
 (1010, 1, 104, 1, 7000.0, 7000.0),
 (1002, 2, 102, 1, 500.0, 500.0),
 (1007, 2, 105, 4, 2500.0, 10000.0),
 (1003, 3, 103, 3, 1200.0, 3600.0),
 (1008, 3, 101, 1, 25000.0, 25000.0),
 (1004, 4, 101, 1, 25000.0, 25000.0),
 (1009, 4, 103, 2, 1200.0, 2400.0),
 (1006, 5, 102, 1, 500.0, 500.0)]

In [18]:
conn = pyodbc.connect(
    "Driver={ODBC Driver 17 for SQL Server};"
    "server=localhost;"
    "Database=Sales_DB;"
    "trusted_connection=yes;"
)
cursor = conn.cursor()
cursor.execute(
"""
create table Sales (
   order_id           int ,
   full_name          varchar(50)  ,
   product_name       varchar(50)  ,
   quantity           int ,
   unit_price         int ,
   total_amount       int 
)
"""
)        

conn.commit()
cursor.close()
conn.close()

In [19]:
conn = pyodbc.connect(
    "Driver={ODBC Driver 17 for SQL Server};"
    "server=localhost;"
    "Database=Sales_DB;"
    "trusted_connection=yes;"
)
cursor = conn.cursor()
cursor.executemany( " Insert Into Sales Values (?,?,?,?,?,?) ",data)        

conn.commit()
cursor.close()
conn.close()

### Flask Application

In [ ]:
from flask import Flask, render_template

app = Flask(__name__)

@app.route("/")
def home():
    conn = pyodbc.connect(
        "Driver={ODBC Driver 17 for SQL Server};"
        "Server=localhost;"
        "Database=Sales_DB;"
        "Trusted_Connection=yes;"
    )
    df = pd.read_sql("SELECT * FROM Sales", conn)
    conn.close()

    return render_template("dashboard.html",data=df)
if __name__ == "__main__":
    app.run(port=5001, debug=True, use_reloader=False)


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit
